In [7]:
import os, json
from umls_python_client import UMLSClient
from rich.console import Console
from rich.markdown import Markdown
from strands import Agent, tool
import dotenv
from strands.agent.conversation_manager import SlidingWindowConversationManager
from strands.agent import AgentResult
from strands.models.openai import OpenAIModel        # for vLLM MedGemma
from strands.models.ollama import OllamaModel        # for Ollama llama3.1
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters
# Initialize the SearchAPI class with your API key
# pip install "strands-agents[openai]" umls-python-client httpx pydantic

import httpx
from typing import Optional, List, Dict, Any, Union
from pydantic import BaseModel, Field

dotenv.load_dotenv()

conversation_manager = SlidingWindowConversationManager(
    window_size=20,  # Maximum number of messages to keep
    should_truncate_results=True, # Enable truncating the tool result when a message is too large for the model's context window 
)

# Environment Variables
umls_api_key = os.environ.get("UMLS_API_KEY")
brave_api_key = os.environ.get("BRAVE_API_KEY")

SYSTEM_PROMPT = "You are a helpful medical and health assistant named Capybara. You provide accurate and reliable information based on your training and available resources. Cite your sources in API format, where appropriate."
ORCHESTRATOR_SYSTEM_PROMPT = "You are a manager agent named Orchestrator that manages multiple agents. One agent is health_agent and it specializes in all things health and exercise related. Your task is to coordinate between them and provide a unified response. If the question is a general knowledge question, answer it to the best of your ability, or indicate you are not certain of the answer."

# Sample / test case section
# Define a query to send to the model
user_query = "What are the consequences of low Vitamin B?"

# --- Pydantic output (nice, typed results back to the agent) ---
class RxNormProps(BaseModel):
    rxcui: str
    name: Optional[str] = None
    synonym: Optional[str] = None
    tty: Optional[str] = None
    status: Optional[str] = None

class MedicationLookupResult(BaseModel):
    query: str
    cui: Optional[str] = None
    preferred_name: Optional[str] = None
    semantic_types: List[str] = Field(default_factory=list)
    rxcui: Optional[str] = None
    rxnorm_properties: Optional[RxNormProps] = None

# --- Helper: pick the best UMLS concept that looks like a drug ---
DRUGY_SEMTYPES = {
    "Pharmacologic Substance",  # T121
    "Clinical Drug",            # T200
    "Antibiotic",               # T195
    "Biomedical or Dental Material",  # sometimes picks formulations
}

def _pick_best_umls_concept(results: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    for it in results.get("result", {}).get("results", []):
        stypes = {st.get("name") for st in it.get("semanticTypes", [])}
        if it.get("ui") not in (None, "NONE") and (stypes & DRUGY_SEMTYPES or it.get("rootSource") == "RXNORM"):
            return it
    # fallback: first concept if nothing matched
    items = results.get("result", {}).get("results", [])
    return items[0] if items else None

# --- RxNav: fetch properties for a given RxCUI (JSON endpoint) ---
def _rxnav_properties(rxcui: str) -> Optional[RxNormProps]:
    # RxNorm API (RxNav) docs: /REST/rxcui/{rxcui}/properties.json
    # https://lhncbc.nlm.nih.gov/RxNav/APIs/api-RxNorm.getRxConceptProperties.html
    url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/properties.json"
    r = httpx.get(url, timeout=30)
    if r.status_code != 200:
        return None
    props = r.json().get("properties") or {}
    return RxNormProps(
        rxcui=str(props.get("rxcui") or rxcui),
        name=props.get("name"),
        synonym=props.get("synonym"),
        tty=props.get("tty"),
        status=props.get("status"),
    )

@tool(name="umls_rxnorm_lookup", description="Find a medication’s UMLS CUI from text, map to RxCUI, and return RxNorm properties.")
def umls_rxnorm_lookup(
    query: str,
    page_size: int = 5
) -> Dict[str, Any]:
    """
    Given a free-text medication query (e.g., 'metformin 500 mg'),
    1) search UMLS for concepts, 2) pick a likely drug concept (CUI),
    3) map to RxCUI via UMLS atoms with SAB='RXNORM', and
    4) fetch RxNorm properties for that RxCUI.

    Returns a dict: {query, cui, preferred_name, semantic_types, rxcui, rxnorm_properties}
    (rxnorm_properties itself is a nested dict or null) so agents can .get() safely.
    """
    api_key = os.getenv("UMLS_API_KEY") or os.getenv("UMLS_APIKEY")
    if not api_key:
        return {"error": "Set UMLS_API_KEY in your environment to use this tool.", "query": query}

    umls = UMLSClient(api_key=api_key)

    # 1) UMLS search → candidate concepts (CUI)
    search = umls.searchAPI.search(
        search_string=query,
        return_id_type="concept",
        search_type="words",
        page_number=1,
        page_size=page_size
    )

    best = _pick_best_umls_concept(search)
    if not best:
        return {"query": query, "cui": None, "preferred_name": None, "semantic_types": [], "rxcui": None, "rxnorm_properties": None}

    cui = best["ui"]
    # Get concept details to read semantic types + preferred name
    info = umls.cuiAPI.get_cui_info(cui=cui)
    pref_name = info["result"].get("name")
    semtypes = [t.get("name") for t in info["result"].get("semanticTypes", [])]

    # 2) UMLS atoms for this CUI, restricted to RXNORM, to get RxCUI codes
    atoms = umls.cuiAPI.get_atoms(cui=cui, sabs="RXNORM")
    rxnorm_codes = sorted({a.get("code") for a in atoms.get("result", []) if a.get("code")})
    rxcui = rxnorm_codes[0] if rxnorm_codes else None

    # 3) RxNav API → properties for that RxCUI
    rxprops = _rxnav_properties(rxcui) if rxcui else None

    return {
        "query": query,
        "cui": cui,
        "preferred_name": pref_name,
        "semantic_types": semtypes,
        "rxcui": rxcui,
        "rxnorm_properties": rxprops.model_dump() if isinstance(rxprops, RxNormProps) else None,
    }

# Normalizer utility to cope with possible representations (dict, pydantic obj, JSON string)
def normalize_med_lookup(result: Union[str, Dict[str, Any], MedicationLookupResult]) -> Dict[str, Any]:
    if isinstance(result, dict):
        return result
    if isinstance(result, MedicationLookupResult):
        base = result.model_dump()
        if base.get("rxnorm_properties") and isinstance(base["rxnorm_properties"], RxNormProps):
            base["rxnorm_properties"] = base["rxnorm_properties"].model_dump()
        return base
    if isinstance(result, str):
        # try JSON parse
        try:
            parsed = json.loads(result)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            return {"raw": result}
    return {"raw": str(result)}

@tool
def pretty_print_markdown(markdown: str) -> None:
    console = Console()
    markdown = Markdown(markup=markdown)
    console.print(markdown)

# Connect to an MCP server using stdio transport
# Note: uvx command syntax differs by platform

# For macOS/Linux:
sequentialthinking_stdio_mcp_client = MCPClient(lambda: stdio_client(
    StdioServerParameters(
        command="docker", 
        args=["run",
        "-i",
        "--rm",
        "mcp/sequentialthinking"]
    )
))

# Create a dictionary for environment variables
env_vars = {
    "BRAVE_API_KEY": brave_api_key,
    "BRAVE_MCP_TRANSPORT": "stdio"
}

# Filter out any keys where the value is None, as the API expects str values
clean_env = {k: v for k, v in env_vars.items() if v is not None}

brave_stdio_mcp_client = MCPClient(lambda: stdio_client(
    StdioServerParameters(
        command="docker", 
        args=[
                "run",
        "-i",
        "--rm",
        "-e",
        "BRAVE_MCP_TRANSPORT",
        "-e",
        "BRAVE_API_KEY",
        "mcp/brave-search"
        ],
        env=clean_env
    )
))

@tool(name="health_agent_tool", description="A health agent that can answer medical queries using MedGemma and UMLS RxNorm lookup.")
def health_agent_tool(query: str) -> AgentResult | None:
    try:
        with sequentialthinking_stdio_mcp_client:
            tools = sequentialthinking_stdio_mcp_client.list_tools_sync()
            tools.append(umls_rxnorm_lookup)
            health_agent = Agent(model=medgemma, system_prompt=SYSTEM_PROMPT, tools=tools, conversation_manager=conversation_manager)
            return health_agent(query)
    except Exception as e:
        print(f"Error occurred: {e}")
        return None

# Models
medgemma = OpenAIModel(
    client_args={"base_url": "http://localhost:8000/v1", "api_key": "not-needed", "timeout": 360},
    model_id="google/medgemma-4b-it",
    params={"stream": True},
)
llama_8b = OllamaModel(
    host="http://localhost:11434",
    model_id="llama3.1:8b",
    max_tokens=1024,
    options={"stream": True, "temperature": 0.1, "top_p": 0.95, "top_k": 40},
)




In [11]:
with sequentialthinking_stdio_mcp_client, brave_stdio_mcp_client:
    tools = sequentialthinking_stdio_mcp_client.list_tools_sync() + brave_stdio_mcp_client.list_tools_sync()
    tools.append(health_agent_tool)
    general_agent = Agent(model=llama_8b, system_prompt=ORCHESTRATOR_SYSTEM_PROMPT, tools=tools, conversation_manager=conversation_manager)
    general_agent("State which model you are named and what is Vitamin A used for in the body?")

2025-08-17 22:28:04,491 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


I am a text-based AI assistant, and I don't have a specific name. However, I can provide information on various topics, including health-related questions.

Vitamin A is an essential nutrient that plays a crucial role in maintaining good vision, immune function, and skin health. It is also important for the proper functioning of the heart, lungs, kidneys, and other organs.

In terms of its specific functions, Vitamin A helps to:

1. Regulate vision: Vitamin A is necessary for the production of rhodopsin, a protein in the retina that allows us to see in low light conditions.
2. Support immune function: Vitamin A helps to protect against infections by supporting the health of white blood cells and other immune system components.
3. Maintain healthy skin: Vitamin A is important for the growth and maintenance of skin cells, which can help to prevent conditions such as acne and psoriasis.
4. Promote bone health: Vitamin A is necessary for the proper absorption of calcium and phosphorus, whi